# NTL: our own ATLAS bundle (Qwen3-8B-Base, 18 source adapters), trained on free Colab (T4)

Runs `name-that-llm/lab/08_train_atlas.py` (embedded below, with our `ntl_atlas` / `ntl_rlcd` package). It trains the
ATLAS method on train.csv only, with the public bundle's recipe (see `lab/atlas_rebuild_spec.md`). Units, in this order:

1. `fold0`: train on 4/5 of the prompts and score the held-out fifth. This is the first honest read, about 3-4 h.
2. `full`: train on all rows and write the drop-in `bundle/` that the 0.636 notebook variant loads, about 3.5-4.5 h.
3. `fold1` .. `fold4`, then `merge`: the full prompt-grouped OOF (`oof_atlas.npz`, `summary_atlas.json`), about 3-4 h each.

**What to do:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Approve the Google Drive
mount, and allow notebook access to the `KAGGLE_API_TOKEN` secret (key icon in the left bar) if Colab asks.
Keep the tab open.

**Resuming.** Training saves a resume point to Google Drive (`MyDrive/ntl_atlas/`, about 1.4 GB) every 10 steps, and
every finished unit is kept there. If Colab disconnects, reconnect and **Run all** again: finished units are skipped and
the running one continues from its last save. To stop after the bundle, set `UNITS = ["fold0", "full"]`.

**Precision.** A T4 cannot hold the 16 GB fp16 base, so the base is loaded 4-bit (NF4) for training and OOF scoring. On a
GPU with at least 30 GB (A100) it uses the exact fp16 recipe. The bundle is scored on Kaggle with the fp16 base on two T4s.

**Results** go to the private Kaggle dataset `aryanawasthiteykik/ntl-atlas-rebuild` after every unit: `bundle/` (when
done), `oof_fold*.npz`, `summary_atlas.json` and `progress.json`.


In [ ]:
OUTPUTS = "aryanawasthiteykik/ntl-atlas-rebuild"   # private, created on the first upload
UNITS = ["fold0", "full", "fold1", "fold2", "fold3", "fold4", "merge"]
MICRO = 8                                  # micro-batch: memory only, the effective batch stays 128
WORK = "/content/drive/MyDrive/ntl_atlas"  # outputs and resume points, survive disconnects
CODE = "/content/ntl_atlas_code"           # the embedded scripts (re-written on every Run all)
DATA = "/content/data"                     # train.csv from the competition
EST_H = {"fold0": 3.5, "full": 4.0, "fold1": 3.5, "fold2": 3.5, "fold3": 3.5, "fold4": 3.5, "merge": 0.0}   # T4 prior


In [ ]:
# A GPU first, so a CPU runtime fails at once with the fix.
import shutil, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No GPU in this runtime. Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.")
GPU = gpu.stdout.strip().splitlines()[0]
GPU_MB = int(GPU.split(",")[1])
QUANT = [] if GPU_MB >= 30000 else ["--quant", "nf4"]   # fp16 base needs >= 30 GB to train
print("GPU:", GPU, "| base:", "nf4" if QUANT else "fp16")


In [ ]:
import json, os, sys, time
from importlib.metadata import PackageNotFoundError, version

from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
except Exception as e:
    raise RuntimeError("Add your Kaggle API token as the Colab secret KAGGLE_API_TOKEN (key icon in the left bar), "
                       "turn on its notebook access, then Run all again.") from e


def ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


need = [] if ver("transformers") == "5.18.0" else ["transformers==5.18.0"]   # the version 08 was checked with
need += [] if int((ver("kaggle") or "0").split(".")[0]) >= 2 else ["kaggle==2.2.4"]
need += [p for p in ["accelerate", "bitsandbytes"] if ver(p) is None]
if need:
    print("pip install", *need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
sh = lambda cmd: subprocess.run(cmd, shell=True, check=True)
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{DATA}/train.csv"):
    sh(f"kaggle competitions download name-that-llm -f train.csv -p {DATA}")
    sh(f"cd {DATA} && (unzip -q -o train.csv.zip && rm -f train.csv.zip || true)")
print("train.csv", os.path.getsize(f"{DATA}/train.csv"), "bytes | outputs on Drive in", WORK)


In [ ]:
# The scripts, embedded verbatim from the repo (name-that-llm/lab); CODE_SHA identifies this version.
import hashlib, os
FILES = {}
FILES['08_train_atlas.py'] = '"""Step 8b: train our own ATLAS bundle (source-conditioned likelihood, arXiv:2610.19583) with prompt-grouped OOF (GPU).\n\nWhat: Qwen3-8B-Base frozen; 18 sources, each with a LoRA up-projection B[s] on q/k/v/o of all 36 layers (A shared, rank 16,\nalpha 32) and a one-token prefix; trained on train.csv only with next-token NLL under the passage\'s own source. The\nrecipe is josephadamski91/ntl-atlas-qwen3-8b-v1\'s training_config (5 epochs, batch 128, AdamW 1e-4, wd 0.01, 5% warm-up,\nclip 1.0, seed 17); its training code is not published, so the gaps are our guesses, listed in atlas_rebuild_spec.md.\nThe model code is our package lab/atlas (ntl_atlas, ntl_rlcd), shipped inside the bundle.\nUnits: fold<k> trains on 4/5 of the prompts (GroupKFold(5) on prompt_id, the same folds as 07) and scores the held-out\nfifth; full trains on all rows and writes the bundle; merge joins the folds.\n\nWhy: the 0.636 notebook takes 10% from ATLAS on uncertain rows. With a bundle we trained ourselves, a variant of that\nnotebook can be a final pick, and the OOF gives ATLAS\'s standalone balanced accuracy, which the public bundle never had.\n\nOutput (in --out):\n- bundle/: the drop-in bundle (atlas_bundle.json, adapters.safetensors, ntl_atlas/, ntl_rlcd/), the only\n  atlas_bundle.json in --out, because the notebook asserts exactly one under /kaggle/input.\n- oof_fold<k>.npz: rows, ids, y, fold, probs and raw evidence (labels order) of the held-out prompts, plus diagnostics.\n- oof_atlas.npz + summary_atlas.json after merge: OOF probabilities of every train row, balanced accuracy overall and\n  per fold.\n- state/<unit>.pt: the resume point (every --save-every steps), deleted when the unit is done.\nA unit whose output exists is skipped (the cache).\n\nUsage:\n  python 08_train_atlas.py --unit fold0 --train train.csv --base Qwen/Qwen3-8B-Base --out OUT [--quant nf4]\n  units: fold0 .. fold4, full, merge, or all (= fold0, full, fold1 .. fold4, merge)\n  CPU smoke test: --allow-cpu --base <tiny dir> --base-dtype float32 --limit 72 --epochs 2 --batch 16\n"""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import balanced_accuracy_score\nfrom sklearn.model_selection import GroupKFold\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(HERE, \'atlas\'))\nfrom ntl_atlas.model import load_model  # noqa: E402\nfrom ntl_rlcd.data import LABELS  # noqa: E402\n\nK = len(LABELS)\nCFG = dict(   # the public bundle\'s training_config, field by field; only these keys reach our model\n    base_model=\'Qwen/Qwen3-8B-Base\', base_revision=\'49e3418fbbbca6ecbdf9608b4d22e5a407081db4\', seed=17,\n    rank=16, alpha=32, dropout=0.05, source_b_init_std=1e-4, learning_rate=1e-4, weight_decay=0.01, epochs=5,\n    effective_batch_size=128, micro_batch_size=4, max_length=512, grad_clip=1.0, warmup_ratio=0.05,\n    source_chunk_size=2, head_chunk_tokens=64, prod_temperatures=[1.0, 2.0, 4.0, 8.0], base_dtype=\'float16\',\n    adapter_dtype=\'float32\')\nPKG = [\'ntl_atlas/__init__.py\', \'ntl_atlas/model.py\', \'ntl_atlas/inference.py\', \'ntl_rlcd/__init__.py\', \'ntl_rlcd/data.py\']\n\n\ndef log(*a):\n    print(time.strftime(\'%H:%M:%S\'), *a, flush=True)\n\n\ndef sha(path):\n    return hashlib.sha256(open(path, \'rb\').read()).hexdigest()\n\n\ndef batches(n, cfg, seed):\n    """Optimizer steps as lists of row positions: a fresh permutation per epoch cut into effective batches; the last batch\n    of an epoch is partial (2699 rows -> 22 steps per epoch, 110 in all, as in the public bundle\'s completed_steps)."""\n    out, bs = [], cfg[\'effective_batch_size\']\n    for e in range(cfg[\'epochs\']):\n        p = np.random.default_rng(seed + e).permutation(n)\n        out += [p[s:s + bs] for s in range(0, n, bs)]\n    return out\n\n\ndef pad(tok, seqs, dev):\n    pk = tok.pad({\'input_ids\': seqs}, padding=True, return_tensors=\'pt\')\n    return pk[\'input_ids\'].to(dev), pk[\'attention_mask\'].to(dev)\n\n\ndef score_rows(model, tok, ids, rows, bs, chunk):\n    """probs and raw evidence (len(rows), K) for token lists ids[rows], shortest first."""\n    rows = np.asarray(rows)\n    order = rows[np.argsort([len(ids[i]) for i in rows], kind=\'stable\')]\n    pos = {r: i for i, r in enumerate(rows)}\n    P, R = np.zeros((len(rows), K), np.float32), np.zeros((len(rows), K), np.float32)\n    for s in range(0, len(order), bs):\n        b = order[s:s + bs]\n        x, m = pad(tok, [ids[i] for i in b], model.device)\n        p, r = model.score(x, m, source_chunk=chunk)\n        at = [pos[i] for i in b]\n        P[at], R[at] = p.float().cpu().numpy(), r.float().cpu().numpy()\n    return P, R\n\n\ndef diag_rows(y, rows, per_class=2):\n    """A fixed small probe set: the first `per_class` rows of each class among `rows`."""\n    return np.concatenate([rows[y[rows] == k][:per_class] for k in range(K)])\n\n\ndef train_unit(unit, tr, ev, ids, y, cfg, a, dev):\n    """Train on rows tr; return the model and tokenizer. Resumes from state/<unit>.pt; `ev` rows are the diagnostic probe."""\n    torch.manual_seed(cfg[\'seed\'])\n    model, tok = load_model(a.base, LABELS, cfg, training=True, device=dev, quant=a.quant)\n    params = [p for p in model.parameters() if p.requires_grad]\n    log(f\'{unit}: {sum(p.numel() for p in params) / 1e6:.1f}M trainable params, {len(tr)} rows, quant={a.quant}\')\n    steps = batches(len(tr), cfg, cfg[\'seed\'])\n    total, warm = len(steps), max(1, math.ceil(cfg[\'warmup_ratio\'] * len(steps)))\n    opt = torch.optim.AdamW(params, lr=cfg[\'learning_rate\'], weight_decay=cfg[\'weight_decay\'])\n    sched = torch.optim.lr_scheduler.LambdaLR(  # linear warm-up, then cosine to 0 (schedule shape is our guess)\n        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))\n    fp16 = cfg[\'base_dtype\'] == \'float16\' and dev != \'cpu\'\n    scaler = torch.amp.GradScaler(\'cuda\', init_scale=2.0**10, enabled=fp16)\n    state = os.path.join(a.out, \'state\', f\'{unit}.pt\')\n    start, hist = 0, []\n    if os.path.exists(state):\n        st = torch.load(state, map_location=\'cpu\', weights_only=False)\n        model.load_trainable(st[\'adapters\'])\n        opt.load_state_dict(st[\'opt\'])\n        sched.load_state_dict(st[\'sched\'])\n        scaler.load_state_dict(st[\'scaler\'])\n        start, hist = st[\'step\'], st[\'hist\']\n        log(f\'{unit}: resumed at step {start}/{total}\')\n    t0 = time.time()\n    for step in range(start, total):\n        rows = tr[steps[step]]\n        rows = rows[np.argsort([len(ids[i]) for i in rows], kind=\'stable\')]   # less padding; same gradient\n        tot = 0.0\n        for s in range(0, len(rows), cfg[\'micro_batch_size\']):\n            mb = rows[s:s + cfg[\'micro_batch_size\']]\n            x, m = pad(tok, [ids[i] for i in mb], model.device)\n            loss = model.loss(x, m, torch.as_tensor(y[mb], device=model.device)) * (len(mb) / len(rows))\n            scaler.scale(loss).backward()\n            tot += loss.item()\n        scaler.unscale_(opt)\n        gn = torch.nn.utils.clip_grad_norm_(params, cfg[\'grad_clip\']).item()\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n        sched.step()\n        hist.append(dict(step=step + 1, loss=tot, grad_norm=gn, lr=sched.get_last_lr()[0], scale=scaler.get_scale()))\n        el = time.time() - t0\n        log(f\'{unit} step {step + 1}/{total} loss {tot:.4f} gnorm {gn:.3f} \'\n            f\'({el / (step + 1 - start):.1f}s/step, eta {el / (step + 1 - start) * (total - step - 1) / 60:.0f} min)\')\n        if a.diag_every and (step + 1) % a.diag_every == 0 and step + 1 < total:\n            P, _ = score_rows(model, tok, ids, ev, a.score_batch, a.score_chunk)\n            log(f\'{unit} diag step {step + 1}: acc {np.mean(P.argmax(1) == y[ev]):.3f} on {len(ev)} probe rows\')\n            hist[-1][\'diag_acc\'] = float(np.mean(P.argmax(1) == y[ev]))\n        if (step + 1) % a.save_every == 0 and step + 1 < total:\n            os.makedirs(os.path.dirname(state), exist_ok=True)\n            torch.save(dict(step=step + 1, hist=hist, adapters=model.trainable_state(), opt=opt.state_dict(),\n                            sched=sched.state_dict(), scaler=scaler.state_dict()), state + \'.tmp\')\n            os.replace(state + \'.tmp\', state)   # never a half-written resume point\n    return model, tok, hist, time.time() - t0\n\n\ndef write_bundle(model, cfg, hist, a, seconds, train_sha):\n    from safetensors.torch import save_file\n    out = os.path.join(a.out, \'bundle\')\n    tmp = out + \'.tmp\'\n    if os.path.exists(tmp):\n        shutil.rmtree(tmp)\n    for f in PKG:\n        os.makedirs(os.path.dirname(os.path.join(tmp, f)), exist_ok=True)\n        shutil.copy(os.path.join(HERE, \'atlas\', f), os.path.join(tmp, f))\n    save_file(model.trainable_state(), os.path.join(tmp, \'adapters.safetensors\'))\n    meta = dict(labels=LABELS, training_config=cfg, completed_steps=len(hist), train_seconds=round(seconds),\n                train_sha256=train_sha, train_base_quant=a.quant or \'none\', trainer=\'lab/08_train_atlas.py (ours)\',\n                method=\'ATLAS (arXiv:2610.19583): shared LoRA A, per-source B and prefix; prod readout, T=1,2,4,8\',\n                files={f: sha(os.path.join(tmp, f)) for f in PKG + [\'adapters.safetensors\']}, history=hist)\n    json.dump(meta, open(os.path.join(tmp, \'atlas_bundle.json\'), \'w\'), indent=1)\n    if os.path.exists(out):\n        shutil.rmtree(out)\n    os.replace(tmp, out)\n    return out\n\n\ndef merge(a, train, y, folds):\n    P, R = np.full((len(train), K), np.nan, np.float32), np.full((len(train), K), np.nan, np.float32)\n    for f in range(a.folds):\n        z = np.load(os.path.join(a.out, f\'oof_fold{f}.npz\'))\n        P[z[\'rows\']], R[z[\'rows\']] = z[\'probs\'], z[\'raw\']\n    assert np.isfinite(P).all(), \'a fold is missing\'\n    bacc = balanced_accuracy_score(y, P.argmax(1))\n    fb = [balanced_accuracy_score(y[folds == f], P[folds == f].argmax(1)) for f in range(a.folds)]\n    np.savez_compressed(os.path.join(a.out, \'oof_atlas.npz\'), ids=train.id.astype(str).to_numpy(dtype=str),\n                        prompt_id=train.prompt_id.astype(str).to_numpy(dtype=str), fold=folds, y=y,\n                        labels=np.array(LABELS), probs=P, raw=R)\n    s = dict(oof_balanced_accuracy=bacc, fold_balanced_accuracy=fb, n=len(train),\n             oof_log_loss=float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-9, 1)))))\n    json.dump(s, open(os.path.join(a.out, \'summary_atlas.json\'), \'w\'), indent=1)\n    log(f\'merge: OOF balanced accuracy {bacc:.4f}, per fold {np.round(fb, 4).tolist()}\')\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--unit\', required=True)\n    ap.add_argument(\'--train\', required=True)\n    ap.add_argument(\'--base\', default=CFG[\'base_model\'], help=\'HF id (pinned to base_revision) or local dir\')\n    ap.add_argument(\'--out\', required=True)\n    ap.add_argument(\'--quant\', choices=[\'nf4\'], default=None, help=\'4-bit base for training on one 16 GB GPU\')\n    ap.add_argument(\'--folds\', type=int, default=5)\n    ap.add_argument(\'--micro\', type=int, help=\'micro-batch (memory only; same effective batch)\')\n    ap.add_argument(\'--score-batch\', type=int, default=4)\n    ap.add_argument(\'--score-chunk\', type=int, default=6, help=\'sources per scoring forward pass\')\n    ap.add_argument(\'--save-every\', type=int, default=10)\n    ap.add_argument(\'--diag-every\', type=int, default=44, help=\'probe accuracy every N steps (0 = off)\')\n    ap.add_argument(\'--allow-cpu\', action=\'store_true\')\n    ap.add_argument(\'--base-dtype\', choices=[\'float16\', \'float32\'], help=\'smoke tests only\')\n    ap.add_argument(\'--limit\', type=int, help=\'smoke tests: keep the first N rows\')\n    ap.add_argument(\'--epochs\', type=int)\n    ap.add_argument(\'--batch\', type=int, help=\'effective batch (smoke tests)\')\n    a = ap.parse_args()\n    cfg = dict(CFG)\n    for k, v in dict(micro_batch_size=a.micro, base_dtype=a.base_dtype, epochs=a.epochs, effective_batch_size=a.batch).items():\n        if v is not None:\n            cfg[k] = v\n    if not os.path.isdir(a.base):\n        from huggingface_hub import snapshot_download\n        a.base = snapshot_download(a.base, revision=cfg[\'base_revision\'],\n                                   allow_patterns=[\'*.json\', \'*.safetensors\', \'*.txt\', \'merges.txt\'])\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    if dev == \'cpu\' and not a.allow_cpu:\n        raise SystemExit(\'no GPU (pass --allow-cpu for a smoke test)\')\n    os.makedirs(a.out, exist_ok=True)\n\n    train = pd.read_csv(a.train)\n    train_sha = sha(a.train)\n    if a.limit:\n        train = train.iloc[:a.limit].reset_index(drop=True)\n    y = train.model.map({l: i for i, l in enumerate(LABELS)}).to_numpy()\n    assert not np.isnan(y.astype(float)).any(), \'unknown label\'\n    folds = np.zeros(len(train), int)\n    for f, (_, va) in enumerate(GroupKFold(n_splits=a.folds).split(train, groups=train.prompt_id)):\n        folds[va] = f\n    from transformers import AutoTokenizer\n    ids = AutoTokenizer.from_pretrained(a.base)(train.text.tolist(), truncation=True, max_length=cfg[\'max_length\'])[\'input_ids\']\n\n    units = ([\'fold0\', \'full\'] + [f\'fold{k}\' for k in range(1, a.folds)] + [\'merge\']) if a.unit == \'all\' else [a.unit]\n    for unit in units:\n        if unit == \'merge\':\n            merge(a, train, y, folds)\n            continue\n        done = os.path.join(a.out, \'bundle\', \'atlas_bundle.json\') if unit == \'full\' else os.path.join(a.out, f\'oof_{unit}.npz\')\n        if os.path.exists(done):\n            log(f\'{unit}: cached ({done})\')\n            continue\n        if unit == \'full\':\n            tr, va = np.arange(len(train)), None\n            ev = diag_rows(y, tr)   # in-sample probe: training health, and the nf4-vs-fp16 check rows\n        else:\n            f = int(unit[4:])\n            tr, va = np.where(folds != f)[0], np.where(folds == f)[0]\n            ev = diag_rows(y, va)   # held-out probe: an early OOF read\n        model, tok, hist, secs = train_unit(unit, tr, ev, ids, y, cfg, a, dev)\n        t1 = time.time()\n        if unit == \'full\':\n            Pd, Rd = score_rows(model, tok, ids, ev, a.score_batch, a.score_chunk)\n            hist.append(dict(final_probe_rows=ev.tolist(), final_probe_acc=float(np.mean(Pd.argmax(1) == y[ev])),\n                             final_probe_probs=np.round(Pd, 6).tolist()))\n            log(f\'full: bundle at {write_bundle(model, cfg, hist, a, secs, train_sha)}, \'\n                f\'in-sample probe acc {np.mean(Pd.argmax(1) == y[ev]):.3f}\')\n        else:\n            P, R = score_rows(model, tok, ids, va, a.score_batch, a.score_chunk)\n            bacc = balanced_accuracy_score(y[va], P.argmax(1))\n            np.savez_compressed(done + \'.tmp.npz\', rows=va, ids=train.id.astype(str).to_numpy(dtype=str)[va], y=y[va],\n                                fold=folds[va], labels=np.array(LABELS), probs=P, raw=R,\n                                hist=json.dumps(hist), train_seconds=secs, score_seconds=time.time() - t1,\n                                quant=a.quant or \'none\')\n            os.replace(done + \'.tmp.npz\', done)\n            log(f\'{unit}: held-out balanced accuracy {bacc:.4f} on {len(va)} rows \'\n                f\'(train {secs / 60:.0f} min, score {(time.time() - t1) / 60:.0f} min)\')\n        st = os.path.join(a.out, \'state\', f\'{unit}.pt\')\n        if os.path.exists(st):\n            os.remove(st)\n        del model\n        if dev == \'cuda\':\n            torch.cuda.empty_cache()\n    if a.unit == \'all\' or a.unit.startswith(\'fold\'):\n        if all(os.path.exists(os.path.join(a.out, f\'oof_fold{f}.npz\')) for f in range(a.folds)) and \\\n                not os.path.exists(os.path.join(a.out, \'summary_atlas.json\')):\n            merge(a, train, y, folds)\n\n\nif __name__ == \'__main__\':\n    main()\n'
FILES['atlas/ntl_atlas/__init__.py'] = '"""Our ATLAS rebuild for Name That LLM (see lab/atlas_rebuild_spec.md)."""\n'
FILES['atlas/ntl_atlas/model.py'] = '"""Our ATLAS scorer for Name That LLM: one frozen causal LM turned into 18 source-conditioned LMs (arXiv:2610.19583).\n\nWritten from scratch for this repo (see lab/atlas_rebuild_spec.md). Per source s (one per label):\n- every attention projection (q, k, v, o of every layer) gets a LoRA delta B[s] @ A, where the down-projection A is shared\n  by all sources and the up-projection B[s] belongs to source s;\n- one learned prefix embedding control[s] is put in front of the passage.\nTraining is next-token NLL of a passage under its own source. Scoring computes the per-token log-likelihood under all 18\nsources, then the clipped pairwise ("prod") readout averaged over temperatures, normalised over sources.\n\nThe 0.636 notebook calls exactly: load_model(base, labels, cfg, device=\'auto\') -> (model, tokenizer);\nmodel.load_trainable(state); model.eval(); model.device; model.score(ids, mask, source_chunk=c) -> (probs, raw).\nTrainable tensor names: base.model.layers.<i>.self_attn.<q|k|v|o>_proj.<a|b> and control.\n"""\nimport math\nimport os\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nTARGETS = (\'q_proj\', \'k_proj\', \'v_proj\', \'o_proj\')\n\n\nclass Router:\n    """The source index of every row in the current forward pass, shared by all routed layers."""\n    src = None\n\n\nclass RoutedLoRA(nn.Module):\n    """y = W x + scale * B[src] (A dropout(x)), with W frozen, A shared and B per source (fp32 adapter maths)."""\n\n    def __init__(self, base, n_src, rank, alpha, dropout, init_std, router):\n        super().__init__()\n        self.base, self.router, self.scale = base, router, alpha / rank\n        dev = base.weight.device\n        self.a = nn.Parameter(torch.empty(rank, base.in_features, device=dev, dtype=torch.float32))\n        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))\n        self.b = nn.Parameter(init_std * torch.randn(n_src, base.out_features, rank, device=dev, dtype=torch.float32))\n        self.drop = nn.Dropout(dropout)\n        base.requires_grad_(False)\n\n    def forward(self, x):\n        src = self.router.src\n        if src is None or src.numel() != x.shape[0]:\n            raise ValueError(\'every row needs a source index\')\n        y = self.base(x)\n        low = self.drop(x.float()) @ self.a.t()                          # (n, t, r)\n        up = self.b[src.to(self.b.device)]                                # (n, out, r)\n        return y + (self.scale * torch.bmm(low, up.transpose(1, 2))).to(y.dtype)\n\n\ndef prod_readout(lp, valid, temperatures):\n    """lp (n, S, T) token log-probs per source, valid (n, T) mask -> (n, S) non-negative evidence.\n    For source k and token t the disadvantage is sum_j max(lp_j - lp_k, 0); evidence is the mean over valid tokens of\n    exp(-disadvantage / temp), averaged over the temperatures."""\n    lp = lp.float()\n    dis = torch.relu(lp[:, None, :, :] - lp[:, :, None, :]).sum(2)     # [n, k, t]: sum over j of (lp_j - lp_k)+\n    w = valid.float()[:, None, :]\n    den = w.sum(-1).clamp_min(1)\n    return torch.stack([(torch.exp(-dis / float(t)) * w).sum(-1) / den for t in temperatures]).mean(0)\n\n\ndef to_probs(raw):\n    if not torch.isfinite(raw).all() or (raw < 0).any():\n        raise ValueError(\'ATLAS evidence must be finite and non-negative\')\n    raw = raw.clamp_min(1e-12)\n    return raw / raw.sum(-1, keepdim=True)\n\n\nclass AtlasModel(nn.Module):\n    def __init__(self, base, labels, config):\n        super().__init__()\n        self.base, self.labels, self.config, self.router = base, list(labels), dict(config), Router()\n        base.requires_grad_(False)\n        base.config.use_cache = False\n        n_src, wrapped = len(self.labels), 0\n        for mod in list(base.modules()):\n            for leaf in TARGETS:\n                child = getattr(mod, leaf, None)\n                if isinstance(child, nn.Linear):\n                    setattr(mod, leaf, RoutedLoRA(child, n_src, config[\'rank\'], config[\'alpha\'], config[\'dropout\'],\n                                                  config[\'source_b_init_std\'], self.router))\n                    wrapped += 1\n        if not wrapped:\n            raise ValueError(\'no attention projections found\')\n        emb = base.get_input_embeddings().weight\n        with torch.no_grad():   # prefix init: the mean token embedding plus 1% of the embedding std as noise, per source\n            total = torch.zeros(emb.shape[1], device=emb.device)\n            sq = torch.zeros((), device=emb.device)\n            for rows in emb.split(4096):\n                rows = rows.float()\n                total += rows.sum(0)\n                sq += rows.square().sum()\n            mean = total / emb.shape[0]\n            std = (sq / emb.numel() - mean.square().mean()).clamp_min(0).sqrt()\n            start = mean.expand(n_src, 1, -1) + 0.01 * std * torch.randn(n_src, 1, emb.shape[1], device=emb.device)\n        self.control = nn.Parameter(start.contiguous())\n\n    @property\n    def device(self):\n        return self.control.device\n\n    def trainable_state(self):\n        return {k: p.detach().cpu().contiguous() for k, p in self.named_parameters() if p.requires_grad}\n\n    def load_trainable(self, state):\n        own = dict((k, p) for k, p in self.named_parameters() if p.requires_grad)\n        if set(own) != set(state):\n            raise ValueError(f\'adapter keys differ: {sorted(set(own) ^ set(state))[:5]}\')\n        with torch.no_grad():\n            for k, p in own.items():\n                v = state[k]\n                if v.shape != p.shape or not torch.isfinite(v).all():\n                    raise ValueError(f\'bad adapter tensor {k}\')\n                p.copy_(v.to(p.device, p.dtype))\n\n    def token_logp(self, ids, mask, src):\n        """(n, L-1) log p(x_t | control[src], x_<t) for t = 1..L-1 (the first passage token is not scored)."""\n        self.router.src = src\n        tok = self.base.get_input_embeddings()(ids)\n        pre = self.control[src.to(self.control.device)].to(tok.device, tok.dtype)\n        x = torch.cat([pre, tok], 1)\n        att = torch.cat([torch.ones_like(mask[:, :1]), mask], 1)\n        h = self.base.model(inputs_embeds=x, attention_mask=att, use_cache=False).last_hidden_state\n        h, y = h[:, 1:-1], ids[:, 1:]   # outputs at x_0..x_{L-2} predict x_1..x_{L-1}\n        head = self.base.get_output_embeddings()\n\n        def chunk_lp(hc, yc):\n            logits = head(hc.to(head.weight.device)).float()\n            return -F.cross_entropy(logits.flatten(0, 1), yc.to(logits.device).flatten(), reduction=\'none\').view(yc.shape)\n\n        step, out = self.config[\'head_chunk_tokens\'], []\n        for s in range(0, h.shape[1], step):\n            hc, yc = h[:, s:s + step], y[:, s:s + step]\n            if self.training and torch.is_grad_enabled():\n                out.append(checkpoint(chunk_lp, hc, yc, use_reentrant=False))\n            else:\n                out.append(chunk_lp(hc, yc))\n        return torch.cat(out, 1).to(ids.device)\n\n    def loss(self, ids, mask, src):\n        """Mean over rows of the per-row mean token NLL under the row\'s own source."""\n        lp = self.token_logp(ids, mask, src)\n        v = mask[:, 1:].float()\n        return -((lp * v).sum(1) / v.sum(1).clamp_min(1)).mean()\n\n    @torch.no_grad()\n    def score(self, ids, mask, source_chunk=None):\n        """(probs, raw), both (n, 18) in self.labels order; `source_chunk` sources share one forward pass."""\n        was = self.training\n        self.eval()\n        try:\n            n, S = len(ids), len(self.labels)\n            c = source_chunk or self.config[\'source_chunk_size\']\n            parts = []\n            for s0 in range(0, S, c):\n                srcs = torch.arange(s0, min(S, s0 + c), device=ids.device)\n                lp = self.token_logp(ids.repeat(len(srcs), 1), mask.repeat(len(srcs), 1), srcs.repeat_interleave(n))\n                parts.append(lp.view(len(srcs), n, -1))\n            raw = prod_readout(torch.cat(parts).transpose(0, 1), mask[:, 1:], self.config[\'prod_temperatures\'])\n            return to_probs(raw), raw\n        finally:\n            self.train(was)\n\n\ndef load_model(path, labels, config, training=False, device=\'cuda\', quant=None):\n    """Frozen base + fresh adapters, and the base tokenizer (right padding). device=\'auto\' spreads the fp16 base over all\n    GPUs (two T4s at inference) and fails if any of it lands off-GPU. quant=\'nf4\' loads the base 4-bit (training on one T4)."""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(path)\n    tok.padding_side = \'right\'\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    dtype = getattr(torch, config.get(\'base_dtype\', \'float16\'))\n    kw = dict(dtype=dtype, attn_implementation=\'sdpa\')\n    gpu = torch.cuda.is_available()\n    if device == \'auto\' and not gpu and os.environ.get(\'NTL_ATLAS_ALLOW_CPU\') != \'1\':\n        raise RuntimeError(\'ATLAS needs a GPU\')\n    if device == \'auto\' and gpu:\n        kw[\'max_memory\'] = {i: int(torch.cuda.get_device_properties(i).total_memory - 3 * 2**30)\n                            for i in range(torch.cuda.device_count())}\n    if device != \'cpu\' and (device != \'auto\' or gpu):\n        kw[\'device_map\'] = device\n    if quant == \'nf4\':\n        from transformers import BitsAndBytesConfig\n        kw[\'quantization_config\'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                                       bnb_4bit_compute_dtype=dtype)\n    base = AutoModelForCausalLM.from_pretrained(path, **kw)\n    if device == \'auto\' and gpu and any(p.device.type != \'cuda\' for p in base.parameters()):\n        raise RuntimeError(\'the fp16 base does not fit on the GPUs (two T4s or one >= 24 GB GPU needed)\')\n    model = AtlasModel(base, labels, config)\n    if training:\n        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n        model.train()\n    else:\n        model.eval()\n    return model, tok\n'
FILES['atlas/ntl_atlas/inference.py'] = '"""predict(texts, base_path, bundle_path, batch_size=1) -> (n, 18) float32 probabilities in ntl_rlcd.data.LABELS order."""\nimport gc\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom safetensors.torch import load_file\n\nfrom ntl_rlcd.data import LABELS\nfrom .model import load_model\n\n\n@torch.inference_mode()\ndef predict(texts, base_path, bundle_path, batch_size=1, source_chunk=1):\n    root = Path(bundle_path)\n    meta = json.loads((root / \'atlas_bundle.json\').read_text())\n    if meta[\'labels\'] != LABELS:\n        raise ValueError(\'bundle label order differs from ntl_rlcd.data.LABELS\')\n    cfg = meta[\'training_config\']\n    model, tok = load_model(base_path, LABELS, cfg, device=\'auto\')\n    try:\n        model.load_trainable(load_file(str(root / \'adapters.safetensors\')))\n        ids = tok(list(texts), truncation=True, max_length=cfg[\'max_length\'])[\'input_ids\']\n        order = np.argsort([len(t) for t in ids], kind=\'stable\')   # shortest first: least padding\n        out = np.zeros((len(ids), len(LABELS)), np.float32)\n        for s in range(0, len(order), batch_size):\n            rows = order[s:s + batch_size]\n            pack = tok.pad({\'input_ids\': [ids[i] for i in rows]}, padding=True, return_tensors=\'pt\').to(model.device)\n            out[rows] = model.score(pack[\'input_ids\'], pack[\'attention_mask\'], source_chunk=source_chunk)[0].float().cpu().numpy()\n        if not np.isfinite(out).all() or not np.allclose(out.sum(1), 1, atol=1e-4):\n            raise RuntimeError(\'invalid ATLAS probabilities\')\n        return out\n    finally:\n        del model\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n'
FILES['atlas/ntl_rlcd/__init__.py'] = '"""Label order shared by our ATLAS bundle and its loader (package name kept for the notebooks\' import)."""\n'
FILES['atlas/ntl_rlcd/data.py'] = '"""The 18 competition labels in the order of the bundle\'s adapters (alphabetical)."""\nLABELS = sorted([\n    \'claude-fable-5\', \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\',\n    \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n    \'glm-5.3\', \'glm-5.3-flash\', \'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n    \'kimi-k3\', \'qwen3.8-max\',\n])\n'
for name, text in FILES.items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
CODE_SHA = 'f4ab2b4bd894678c'
print("wrote", len(FILES), "files to", CODE, "| code", CODE_SHA)


In [ ]:
import shutil

UP = "/content/upload"
log_lines = []


def upload(msg, **progress):
    """Upload the finished outputs (no resume points) and progress.json as a new version of OUTPUTS; 3 tries."""
    if os.path.exists(UP):
        shutil.rmtree(UP)
    os.makedirs(UP)
    for f in sorted(os.listdir(WORK)):
        src = f"{WORK}/{f}"
        if f == "bundle":
            shutil.copytree(src, f"{UP}/bundle")
        elif f.endswith((".npz", ".json")) and not f.endswith(".tmp.npz"):
            shutil.copy(src, UP)
    progress.update(event=msg, gpu=GPU, quant=QUANT, code_sha256=CODE_SHA,
                    time_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
    json.dump(progress, open(f"{UP}/progress.json", "w"), indent=1)
    json.dump({"title": OUTPUTS.split("/")[1], "id": OUTPUTS, "licenses": [{"name": "CC-BY-SA-4.0"}]},
              open(f"{UP}/dataset-metadata.json", "w"))
    for attempt in range(3):
        exists = subprocess.run(["kaggle", "datasets", "status", OUTPUTS], capture_output=True).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", UP, "-m", msg, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", UP, "-r", "zip"])
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"uploaded ({msg}) to https://www.kaggle.com/datasets/{OUTPUTS}", flush=True)
            return
        print(f"upload attempt {attempt + 1} failed: {(r.stdout + r.stderr).strip()[-300:]}", flush=True)
        time.sleep(60)   # a just-created dataset takes a moment before it accepts versions
    print("WARNING: upload failed; the files are safe on Drive in", WORK, flush=True)


def done(unit):
    if unit == "merge":
        return os.path.exists(f"{WORK}/summary_atlas.json")
    return os.path.exists(f"{WORK}/bundle/atlas_bundle.json" if unit == "full" else f"{WORK}/oof_{unit}.npz")


In [ ]:
hours = {}
for unit in UNITS:
    if done(unit):
        print(unit, "already done")
        continue
    left = sum(EST_H[u] for u in UNITS if not done(u))
    print(f"\n=== {unit} (about {EST_H[unit]} h on a T4; {left:.1f} h of units left)", flush=True)
    t = time.time()
    cmd = [sys.executable, "-u", f"{CODE}/08_train_atlas.py", "--unit", unit, "--train", f"{DATA}/train.csv",
           "--out", WORK, "--micro", str(MICRO), *QUANT]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false"})
    for line in p.stdout:
        if "warn" not in line.lower():
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{unit} failed (exit {p.returncode}); Run all again to resume from the last save")
    hours[unit] = round((time.time() - t) / 3600, 2)
    summary = json.load(open(f"{WORK}/summary_atlas.json")) if os.path.exists(f"{WORK}/summary_atlas.json") else None
    upload(f"{unit} done", units_done=[u for u in UNITS if done(u)], hours_this_session=hours, summary=summary)
print("all units done:", [u for u in UNITS if done(u)])
if os.path.exists(f"{WORK}/summary_atlas.json"):
    print(open(f"{WORK}/summary_atlas.json").read())
